In [ ]:

!pip -q install sentence-transformers beautifulsoup4 requests pypdf pandas numpy


In [ ]:

import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import os
import json
import time

base_url = "https://scikit-learn.org/stable/"
guide_url = base_url + "user_guide.html"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(guide_url, headers=headers, timeout=30)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

# Find documentation page links
links = []

for a in soup.find_all("a", href=True):
    url = urljoin(guide_url, a["href"]).split("#")[0]
    parsed = urlparse(url)

    if (
        parsed.netloc == "scikit-learn.org"
        and "/stable/" in parsed.path
        and parsed.path.endswith(".html")
        and url != guide_url
    ):
        if url not in links:
            links.append(url)

print("Documentation links found:", len(links))

# Extract text from documentation pages
documents = []

for url in links:
    try:
        r = requests.get(url, headers=headers, timeout=20)

        if r.status_code != 200:
            continue

        page = BeautifulSoup(r.text, "html.parser")

        main = page.find("main") or page.find("article")

        if main:
            for tag in main(["script", "style", "nav", "footer"]):
                tag.decompose()

            content = main.get_text(" ", strip=True)

            if len(content.split()) > 100:
                documents.append({
                    "url": url,
                    "title": page.title.get_text(strip=True) if page.title else url,
                    "text": content
                })

        if len(documents) >= 40:
            break

        time.sleep(0.2)

    except Exception as e:
        print("Skipped:", url, str(e)[:100])

print("\nSuccessfully collected pages:", len(documents))
print("Total words:", sum(len(d["text"].split()) for d in documents))

os.makedirs("/content/task2_data", exist_ok=True)

with open("/content/task2_data/documents.json", "w", encoding="utf-8") as f:
    json.dump(documents, f, ensure_ascii=False, indent=2)

print("\nDocuments saved successfully!")


Documentation links found: 80

Successfully collected pages: 40
Total words: 122834

Documents saved successfully!


In [ ]:

import json
import os

with open("/content/task2_data/documents.json", "r", encoding="utf-8") as f:
    documents = json.load(f)

def create_chunks(documents, chunk_size, overlap):
    chunks = []

    for doc_id, doc in enumerate(documents):
        words = doc["text"].split()
        step = chunk_size - overlap

        for start in range(0, len(words), step):
            end = min(start + chunk_size, len(words))

            chunk_text = " ".join(words[start:end])

            if len(chunk_text.split()) < 30:
                continue

            chunks.append({
                "chunk_id": len(chunks),
                "doc_id": doc_id,
                "title": doc["title"],
                "url": doc["url"],
                "text": chunk_text,
                "word_count": len(chunk_text.split())
            })

            if end == len(words):
                break

    return chunks


small_chunks = create_chunks(documents, 200, 40)
large_chunks = create_chunks(documents, 500, 80)

print("Small chunks (200 words):", len(small_chunks))
print("Large chunks (500 words):", len(large_chunks))

print("\nExample small chunk:")
print(small_chunks[0]["text"][:500])

print("\nSource URL:")
print(small_chunks[0]["url"])

os.makedirs("/content/task2_data/chunks", exist_ok=True)

with open("/content/task2_data/chunks/small_chunks.json", "w", encoding="utf-8") as f:
    json.dump(small_chunks, f, ensure_ascii=False, indent=2)

with open("/content/task2_data/chunks/large_chunks.json", "w", encoding="utf-8") as f:
    json.dump(large_chunks, f, ensure_ascii=False, indent=2)

print("\nBoth chunk datasets saved successfully!")


Small chunks (200 words): 782
Large chunks (500 words): 305

Example small chunk:
Classification Identifying which category an object belongs to. Applications: Spam detection, image recognition. Algorithms: Gradient boosting , nearest neighbors , random forest , logistic regression , and more... Examples Regression Predicting a continuous-valued attribute associated with an object. Applications: Drug response, stock prices. Algorithms: Gradient boosting , nearest neighbors , random forest , ridge , and more... Examples Clustering Automatic grouping of similar objects into set

Source URL:
https://scikit-learn.org/stable/index.html

Both chunk datasets saved successfully!


In [ ]:

from sentence_transformers import SentenceTransformer
import numpy as np
import os

# Load embedding model
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

# Extract text from chunks
small_texts = [chunk["text"] for chunk in small_chunks]
large_texts = [chunk["text"] for chunk in large_chunks]

# Generate embeddings
small_embeddings = embedding_model.encode(
    small_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

large_embeddings = embedding_model.encode(
    large_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

# Save embeddings
os.makedirs("/content/task2_data/embeddings", exist_ok=True)

np.save(
    "/content/task2_data/embeddings/small_embeddings.npy",
    small_embeddings
)

np.save(
    "/content/task2_data/embeddings/large_embeddings.npy",
    large_embeddings
)

print("Small embedding shape:", small_embeddings.shape)
print("Large embedding shape:", large_embeddings.shape)
print("Embedding generation completed!")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (6 outputs left out)

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/25 [00:00<?, ?it/s]

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Small embedding shape: (782, 384)
Large embedding shape: (305, 384)
Embedding generation completed!


In [ ]:

def retrieve(query, chunks, embeddings, top_k=5):
    # Convert question into embedding
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    # Calculate cosine similarity
    scores = np.dot(embeddings, query_embedding)

    # Get top matching chunk indexes
    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for idx in top_indices:
        results.append({
            "score": float(scores[idx]),
            "title": chunks[idx]["title"],
            "url": chunks[idx]["url"],
            "text": chunks[idx]["text"]
        })

    return results


# Test the retrieval system
query = "What is logistic regression and how is it used for classification?"

results = retrieve(
    query,
    small_chunks,
    small_embeddings,
    top_k=5
)

for i, result in enumerate(results, 1):
    print(f"\nResult {i}")
    print("Similarity:", round(result["score"], 4))
    print("Title:", result["title"])
    print("Source:", result["url"])
    print("Passage:", result["text"][:400])
    print("-" * 70)



Result 1
Similarity: 0.647
Title: 1.1. Linear Models — scikit-learn 1.9.1 documentation
Source: https://scikit-learn.org/stable/modules/linear_model.html
Passage: , \(\ell_2\) or Elastic-Net regularization. Note Regularization Regularization is applied by default, which is common in machine learning but not in statistics. Another advantage of regularization is that it improves numerical stability. No regularization amounts to setting C to a very high value. Note Logistic Regression as a special case of the Generalized Linear Models (GLM) Logistic regression
----------------------------------------------------------------------

Result 2
Similarity: 0.5729
Title: 1.1. Linear Models — scikit-learn 1.9.1 documentation
Source: https://scikit-learn.org/stable/modules/linear_model.html
Passage: 1.1. Linear Models # The following are a set of methods intended for regression in which the target value is expected to be a linear combination of the features. In mathematical notation, the predict

In [ ]:

# Display all collected documentation pages

for i, doc in enumerate(documents, 1):
    print(f"{i}. {doc['title']}")
    print("URL:", doc["url"])
    print("-" * 70)


1. scikit-learn: machine learning in Python — scikit-learn 1.9.1 documentation
URL: https://scikit-learn.org/stable/index.html
----------------------------------------------------------------------
2. Installing scikit-learn — scikit-learn 1.9.1 documentation
URL: https://scikit-learn.org/stable/install.html
----------------------------------------------------------------------
3. API Reference — scikit-learn 1.9.1 documentation
URL: https://scikit-learn.org/stable/api/index.html
----------------------------------------------------------------------
4. Examples — scikit-learn 1.9.1 documentation
URL: https://scikit-learn.org/stable/auto_examples/index.html
----------------------------------------------------------------------
5. Getting Started — scikit-learn 1.9.1 documentation
URL: https://scikit-learn.org/stable/getting_started.html
----------------------------------------------------------------------
6. Release History — scikit-learn 1.9.1 documentation
URL: https://scikit-learn.o

In [ ]:

questions = [
    {
        "id": 1,
        "question": "What is the purpose of linear regression?",
        "source": "linear_model.html"
    },
    {
        "id": 2,
        "question": "How does logistic regression perform classification?",
        "source": "linear_model.html"
    },
    {
        "id": 3,
        "question": "What is the main idea behind Support Vector Machines?",
        "source": "svm.html"
    },
    {
        "id": 4,
        "question": "How does the nearest neighbors algorithm make predictions?",
        "source": "neighbors.html"
    },
    {
        "id": 5,
        "question": "What is the basic assumption of the Naive Bayes classifier?",
        "source": "naive_bayes.html"
    },
    {
        "id": 6,
        "question": "How does a decision tree select a feature for splitting?",
        "source": "tree.html"
    },
    {
        "id": 7,
        "question": "What is the difference between bagging and boosting?",
        "source": "ensemble.html"
    },
    {
        "id": 8,
        "question": "What is feature selection and why is it useful?",
        "source": "feature_selection.html"
    },
    {
        "id": 9,
        "question": "What is semi-supervised learning?",
        "source": "semi_supervised.html"
    },
    {
        "id": 10,
        "question": "Why is probability calibration useful in classification?",
        "source": "calibration.html"
    },
    {
        "id": 11,
        "question": "What is the difference between supervised and unsupervised neural network models?",
        "source": "neural_networks_supervised.html"
    },
    {
        "id": 12,
        "question": "What is clustering and what is its purpose?",
        "source": "clustering.html"
    },
    {
        "id": 13,
        "question": "What is a Gaussian mixture model?",
        "source": "mixture.html"
    },
    {
        "id": 14,
        "question": "What is manifold learning used for?",
        "source": "manifold.html"
    },
    {
        "id": 15,
        "question": "How does Isolation Forest help detect outliers?",
        "source": "outlier_detection.html"
    }
]

def show_results(question_list, chunks, embeddings, label):
    print("\n", "=" * 75)
    print(label)
    print("=" * 75)

    for q in question_list:
        results = retrieve(q["question"], chunks, embeddings, top_k=5)

        print(f"\nQ{q['id']}: {q['question']}")
        print("Expected source:", q["source"])

        for rank, result in enumerate(results, 1):
            print(
                f"\nTop {rank} | Score: {result['score']:.4f}"
            )
            print("URL:", result["url"])
            print("Passage:", result["text"][:220])

        print("-" * 65)


show_results(
    questions,
    small_chunks,
    small_embeddings,
    "200-WORD CHUNK RESULTS"
)

show_results(
    questions,
    large_chunks,
    large_embeddings,
    "500-WORD CHUNK RESULTS"
)



200-WORD CHUNK RESULTS

Q1: What is the purpose of linear regression?
Expected source: linear_model.html

Top 1 | Score: 0.4089
URL: https://scikit-learn.org/stable/modules/linear_model.html
Passage: for two-dimensional data: \[\hat{y}(w, x) = w_0 + w_1 x_1 + w_2 x_2\] If we want to fit a paraboloid to the data instead of a plane, we can combine the features in second-order polynomials, so that the model looks like t

Top 2 | Score: 0.3987
URL: https://scikit-learn.org/stable/modules/linear_model.html
Passage: intercept_ attributes: >>> from sklearn import linear_model >>> reg = linear_model . LinearRegression () >>> reg . fit ([[ 0 , 0 ], [ 1 , 1 ], [ 2 , 2 ]], [ 0 , 1 , 2 ]) LinearRegression() >>> reg . coef_ array([0.5, 0.5

Top 3 | Score: 0.3921
URL: https://scikit-learn.org/stable/modules/linear_model.html
Passage: in the residuals, quantile regression is much more robust to outliers than squared error based estimation of the mean. Somewhat in between is the HuberRegressor . Refe

In [ ]:

for doc in documents:
    if "neural_networks_supervised.html" in doc["url"]:
        print("DOCUMENT:", doc["title"])
        print("SOURCE:", doc["url"])
        print("\nDOCUMENT EXCERPT:\n")
        print(doc["text"][:3000])
        break


DOCUMENT: 1.17. Neural network models (supervised) — scikit-learn 1.9.1 documentation
SOURCE: https://scikit-learn.org/stable/modules/neural_networks_supervised.html

DOCUMENT EXCERPT:

1.17. Neural network models (supervised) # Warning This implementation is not intended for large-scale applications. In particular,
scikit-learn offers no GPU support. For much faster, GPU-based implementations,
as well as frameworks offering much more flexibility to build deep learning
architectures, see Related Projects . 1.17.1. Multi-layer Perceptron # Multi-layer Perceptron (MLP) is a supervised learning algorithm that learns
a function \(f: R^m \rightarrow R^o\) by training on a dataset,
where \(m\) is the number of dimensions for input and \(o\) is the
number of dimensions for output. Given a set of features \(X = \{x_1, x_2, ..., x_m\}\) and a target \(y\) , it can learn a non-linear function approximator for either
classification or regression. It is different from logistic regression, in that


In [ ]:
questions[10]["question"] = (
    "What is a Multi-layer Perceptron (MLP), and how can it be used "
    "for classification or regression?"
)

for q in [questions[10]]:
    print("QUESTION:", q["question"])

    for name, chunks, embeddings in [
        ("200-word chunks", small_chunks, small_embeddings),
        ("500-word chunks", large_chunks, large_embeddings)
    ]:
        print("\n", name)

        results = retrieve(q["question"], chunks, embeddings, top_k=5)

        for i, r in enumerate(results, 1):
            print(f"\nRank {i} | Score: {r['score']:.4f}")
            print("Title:", r["title"])
            print("URL:", r["url"])
            print("Passage:", r["text"][:500])

QUESTION: What is a Multi-layer Perceptron (MLP), and how can it be used for classification or regression?

 200-word chunks

Rank 1 | Score: 0.6809
Title: 1.17. Neural network models (supervised) — scikit-learn 1.9.1 documentation
URL: https://scikit-learn.org/stable/modules/neural_networks_supervised.html
Passage: 1.17. Neural network models (supervised) # Warning This implementation is not intended for large-scale applications. In particular, scikit-learn offers no GPU support. For much faster, GPU-based implementations, as well as frameworks offering much more flexibility to build deep learning architectures, see Related Projects . 1.17.1. Multi-layer Perceptron # Multi-layer Perceptron (MLP) is a supervised learning algorithm that learns a function \(f: R^m \rightarrow R^o\) by training on a dataset, w

Rank 2 | Score: 0.5879
Title: 1.17. Neural network models (supervised) — scikit-learn 1.9.1 documentation
URL: https://scikit-learn.org/stable/modules/neural_networks_supervised.ht

In [ ]:
import json
import os

os.makedirs("/content/task2_data/evaluation", exist_ok=True)

with open("/content/task2_data/evaluation/questions.json", "w") as f:
    json.dump(questions, f, indent=4)

print("Updated 15 questions saved successfully.")
print("Total questions:", len(questions))

Updated 15 questions saved successfully.
Total questions: 15


In [ ]:
import numpy as np

def evaluate_retrieval(questions, chunks, embeddings):
    ranks = []

    for q in questions:
        query_embedding = embedding_model.encode(
            [q["question"]],
            convert_to_numpy=True,
            normalize_embeddings=True
        )[0]

        scores = np.dot(embeddings, query_embedding)
        top_indices = np.argsort(scores)[::-1][:5]

        expected_source = q["source"]

        rank = None

        for i, idx in enumerate(top_indices, 1):
            if expected_source in chunks[idx]["url"]:
                rank = i
                break

        ranks.append(rank)

    results = {
        "Recall@1": sum(r is not None and r <= 1 for r in ranks) / len(ranks),
        "Recall@3": sum(r is not None and r <= 3 for r in ranks) / len(ranks),
        "Recall@5": sum(r is not None and r <= 5 for r in ranks) / len(ranks),
        "MRR": np.mean([1/r if r is not None else 0 for r in ranks])
    }

    return results, ranks


small_results, small_ranks = evaluate_retrieval(
    questions, small_chunks, small_embeddings
)

large_results, large_ranks = evaluate_retrieval(
    questions, large_chunks, large_embeddings
)

print("200-WORD CHUNKS")
for metric, value in small_results.items():
    print(f"{metric}: {value:.4f}")

print("\n500-WORD CHUNKS")
for metric, value in large_results.items():
    print(f"{metric}: {value:.4f}")

print("\nQUESTION-WISE RANKS")
for i, q in enumerate(questions):
    print(f"Q{i+1}: 200-word={small_ranks[i]}, 500-word={large_ranks[i]}")

200-WORD CHUNKS
Recall@1: 0.9333
Recall@3: 1.0000
Recall@5: 1.0000
MRR: 0.9667

500-WORD CHUNKS
Recall@1: 1.0000
Recall@3: 1.0000
Recall@5: 1.0000
MRR: 1.0000

QUESTION-WISE RANKS
Q1: 200-word=1, 500-word=1
Q2: 200-word=1, 500-word=1
Q3: 200-word=1, 500-word=1
Q4: 200-word=1, 500-word=1
Q5: 200-word=1, 500-word=1
Q6: 200-word=2, 500-word=1
Q7: 200-word=1, 500-word=1
Q8: 200-word=1, 500-word=1
Q9: 200-word=1, 500-word=1
Q10: 200-word=1, 500-word=1
Q11: 200-word=1, 500-word=1
Q12: 200-word=1, 500-word=1
Q13: 200-word=1, 500-word=1
Q14: 200-word=1, 500-word=1
Q15: 200-word=1, 500-word=1


In [ ]:
import json
import os

evaluation_data = {
    "dataset": "Scikit-learn User Guide",
    "total_questions": len(questions),
    "metrics": {
        "200_word_chunks": {
            k: float(v) for k, v in small_results.items()
        },
        "500_word_chunks": {
            k: float(v) for k, v in large_results.items()
        }
    },
    "question_wise_ranks": [
        {
            "question": q["question"],
            "200_word_rank": small_ranks[i],
            "500_word_rank": large_ranks[i]
        }
        for i, q in enumerate(questions)
    ]
}

path = "/content/task2_data/evaluation/retrieval_metrics.json"

with open(path, "w") as f:
    json.dump(evaluation_data, f, indent=4)

print("Evaluation results saved:", path)

Evaluation results saved: /content/task2_data/evaluation/retrieval_metrics.json


In [ ]:
import json
import os

failures = [
    {
        "question": 6,
        "topic": "Decision trees",
        "issue": "With 200-word chunks, an ensemble-learning passage ranked above the relevant decision-tree passage.",
        "improvement": "Use heading-aware chunking and combine semantic retrieval with BM25 keyword retrieval."
    },
    {
        "question": 14,
        "topic": "Manifold learning",
        "issue": "A passage containing unrelated lyric-like text received a high similarity score.",
        "improvement": "Improve HTML extraction by retaining main article content and removing irrelevant text."
    },
    {
        "question": 15,
        "topic": "Isolation Forest",
        "issue": "Some highly ranked passages discussed general outlier detection or references rather than directly explaining Isolation Forest.",
        "improvement": "Use more specific queries and a cross-encoder reranker to improve passage relevance."
    }
]

path = "/content/task2_data/evaluation/failure_analysis.json"

with open(path, "w") as f:
    json.dump(failures, f, indent=4)

print("Failure analysis saved successfully.")
print("Total examples:", len(failures))

Failure analysis saved successfully.
Total examples: 3


In [ ]:
!pip install -q rank-bm25
print("BM25 installed successfully.")

BM25 installed successfully.


In [ ]:
import re
from rank_bm25 import BM25Okapi

def tokenize(text):
    return re.findall(r"\b[a-zA-Z0-9_]+\b", text.lower())

small_corpus = [tokenize(c["text"]) for c in small_chunks]
large_corpus = [tokenize(c["text"]) for c in large_chunks]

small_bm25 = BM25Okapi(small_corpus)
large_bm25 = BM25Okapi(large_corpus)

print("BM25 indexes created successfully.")
print("200-word chunks:", len(small_corpus))
print("500-word chunks:", len(large_corpus))

BM25 indexes created successfully.
200-word chunks: 782
500-word chunks: 305


In [ ]:
import numpy as np

def evaluate_bm25(questions, chunks, bm25):
    ranks = []

    for q in questions:
        query_tokens = tokenize(q["question"])
        scores = bm25.get_scores(query_tokens)
        top_indices = np.argsort(scores)[::-1][:5]

        rank = None

        for i, idx in enumerate(top_indices, 1):
            if q["source"] in chunks[idx]["url"]:
                rank = i
                break

        ranks.append(rank)

    results = {
        "Recall@1": sum(r is not None and r <= 1 for r in ranks) / len(ranks),
        "Recall@3": sum(r is not None and r <= 3 for r in ranks) / len(ranks),
        "Recall@5": sum(r is not None and r <= 5 for r in ranks) / len(ranks),
        "MRR": np.mean([1/r if r is not None else 0 for r in ranks])
    }

    return results, ranks


small_bm25_results, small_bm25_ranks = evaluate_bm25(
    questions, small_chunks, small_bm25
)

large_bm25_results, large_bm25_ranks = evaluate_bm25(
    questions, large_chunks, large_bm25
)

print("BM25 - 200 WORD CHUNKS")
for k, v in small_bm25_results.items():
    print(f"{k}: {v:.4f}")

print("\nBM25 - 500 WORD CHUNKS")
for k, v in large_bm25_results.items():
    print(f"{k}: {v:.4f}")

print("\nQUESTION-WISE RANKS")
for i in range(len(questions)):
    print(
        f"Q{i+1}: 200-word={small_bm25_ranks[i]}, "
        f"500-word={large_bm25_ranks[i]}"
    )

BM25 - 200 WORD CHUNKS
Recall@1: 0.8667
Recall@3: 0.8667
Recall@5: 1.0000
MRR: 0.8967

BM25 - 500 WORD CHUNKS
Recall@1: 0.8000
Recall@3: 0.8000
Recall@5: 0.9333
MRR: 0.8267

QUESTION-WISE RANKS
Q1: 200-word=5, 500-word=None
Q2: 200-word=1, 500-word=1
Q3: 200-word=1, 500-word=1
Q4: 200-word=1, 500-word=1
Q5: 200-word=1, 500-word=1
Q6: 200-word=1, 500-word=1
Q7: 200-word=1, 500-word=1
Q8: 200-word=4, 500-word=5
Q9: 200-word=1, 500-word=1
Q10: 200-word=1, 500-word=1
Q11: 200-word=1, 500-word=1
Q12: 200-word=1, 500-word=5
Q13: 200-word=1, 500-word=1
Q14: 200-word=1, 500-word=1
Q15: 200-word=1, 500-word=1


In [ ]:
def hybrid_retrieve(query, chunks, embeddings, bm25, top_k=5):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    semantic_scores = np.dot(embeddings, query_embedding)
    semantic_order = np.argsort(semantic_scores)[::-1]

    bm25_scores = bm25.get_scores(tokenize(query))
    bm25_order = np.argsort(bm25_scores)[::-1]

    rrf_scores = {}

    for rank, idx in enumerate(semantic_order, 1):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1 / (60 + rank)

    for rank, idx in enumerate(bm25_order, 1):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1 / (60 + rank)

    ranked_indices = sorted(
        rrf_scores,
        key=lambda idx: rrf_scores[idx],
        reverse=True
    )[:top_k]

    results = []

    for idx in ranked_indices:
        results.append({
            "score": float(rrf_scores[idx]),
            "title": chunks[idx]["title"],
            "url": chunks[idx]["url"],
            "text": chunks[idx]["text"]
        })

    return results


print("Hybrid retrieval function created successfully.")

Hybrid retrieval function created successfully.


In [ ]:
def evaluate_hybrid(questions, chunks, embeddings, bm25):
    ranks = []

    for q in questions:
        results = hybrid_retrieve(
            q["question"], chunks, embeddings, bm25, top_k=5
        )

        rank = None

        for i, result in enumerate(results, 1):
            if q["source"] in result["url"]:
                rank = i
                break

        ranks.append(rank)

    metrics = {
        "Recall@1": sum(r is not None and r <= 1 for r in ranks) / len(ranks),
        "Recall@3": sum(r is not None and r <= 3 for r in ranks) / len(ranks),
        "Recall@5": sum(r is not None and r <= 5 for r in ranks) / len(ranks),
        "MRR": np.mean([1/r if r is not None else 0 for r in ranks])
    }

    return metrics, ranks


hybrid_small, hybrid_small_ranks = evaluate_hybrid(
    questions, small_chunks, small_embeddings, small_bm25
)

hybrid_large, hybrid_large_ranks = evaluate_hybrid(
    questions, large_chunks, large_embeddings, large_bm25
)

print("HYBRID - 200 WORD CHUNKS")
for k, v in hybrid_small.items():
    print(f"{k}: {v:.4f}")

print("\nHYBRID - 500 WORD CHUNKS")
for k, v in hybrid_large.items():
    print(f"{k}: {v:.4f}")

print("\nQUESTION-WISE RANKS")
for i in range(len(questions)):
    print(
        f"Q{i+1}: 200-word={hybrid_small_ranks[i]}, "
        f"500-word={hybrid_large_ranks[i]}"
    )

HYBRID - 200 WORD CHUNKS
Recall@1: 1.0000
Recall@3: 1.0000
Recall@5: 1.0000
MRR: 1.0000

HYBRID - 500 WORD CHUNKS
Recall@1: 0.9333
Recall@3: 1.0000
Recall@5: 1.0000
MRR: 0.9556

QUESTION-WISE RANKS
Q1: 200-word=1, 500-word=3
Q2: 200-word=1, 500-word=1
Q3: 200-word=1, 500-word=1
Q4: 200-word=1, 500-word=1
Q5: 200-word=1, 500-word=1
Q6: 200-word=1, 500-word=1
Q7: 200-word=1, 500-word=1
Q8: 200-word=1, 500-word=1
Q9: 200-word=1, 500-word=1
Q10: 200-word=1, 500-word=1
Q11: 200-word=1, 500-word=1
Q12: 200-word=1, 500-word=1
Q13: 200-word=1, 500-word=1
Q14: 200-word=1, 500-word=1
Q15: 200-word=1, 500-word=1


In [ ]:
import json
import os

comparison = {
    "semantic": {
        "200_word": small_results,
        "500_word": large_results
    },
    "bm25": {
        "200_word": small_bm25_results,
        "500_word": large_bm25_results
    },
    "hybrid": {
        "200_word": hybrid_small,
        "500_word": hybrid_large
    },
    "best_configuration": "Hybrid retrieval with 200-word chunks"
}

path = "/content/task2_data/evaluation/retrieval_comparison.json"

with open(path, "w") as f:
    json.dump(comparison, f, indent=4)

print("Complete retrieval comparison saved.")

Complete retrieval comparison saved.


In [ ]:
!pip install -q sentence-transformers
print("Reranking dependencies are ready.")

Reranking dependencies are ready.


In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

print("Cross-encoder reranker loaded successfully.")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Cross-encoder reranker loaded successfully.


In [ ]:
def rerank_retrieval(query, chunks, embeddings, bm25,
                     candidate_k=10, top_k=5):

    candidates = hybrid_retrieve(
        query, chunks, embeddings, bm25, top_k=candidate_k
    )

    pairs = [
        [query, item["text"]]
        for item in candidates
    ]

    scores = reranker.predict(pairs)

    for item, score in zip(candidates, scores):
        item["rerank_score"] = float(score)

    candidates.sort(
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    return candidates[:top_k]


print("Reranking function created successfully.")

Reranking function created successfully.


In [ ]:
query = "What is a Multi-layer Perceptron (MLP), and how can it be used for classification or regression?"

results = rerank_retrieval(
    query,
    small_chunks,
    small_embeddings,
    small_bm25
)

for i, item in enumerate(results, 1):
    print(f"\nRank {i}")
    print("Reranking score:", round(item["rerank_score"], 4))
    print("Title:", item["title"])
    print("URL:", item["url"])
    print("Passage:", item["text"][:500])


Rank 1
Reranking score: 5.045
Title: 1.17. Neural network models (supervised) — scikit-learn 1.9.1 documentation
URL: https://scikit-learn.org/stable/modules/neural_networks_supervised.html
Passage: 1.17. Neural network models (supervised) # Warning This implementation is not intended for large-scale applications. In particular, scikit-learn offers no GPU support. For much faster, GPU-based implementations, as well as frameworks offering much more flexibility to build deep learning architectures, see Related Projects . 1.17.1. Multi-layer Perceptron # Multi-layer Perceptron (MLP) is a supervised learning algorithm that learns a function \(f: R^m \rightarrow R^o\) by training on a dataset, w

Rank 2
Reranking score: 4.9202
Title: 1.17. Neural network models (supervised) — scikit-learn 1.9.1 documentation
URL: https://scikit-learn.org/stable/modules/neural_networks_supervised.html
Passage: real-time (on-line learning) using partial_fit . The disadvantages of Multi-layer Perceptron (MLP)

In [ ]:
import numpy as np

def evaluate_reranker(questions, chunks, embeddings, bm25):
    ranks = []

    for q in questions:
        results = rerank_retrieval(
            q["question"],
            chunks,
            embeddings,
            bm25,
            candidate_k=10,
            top_k=5
        )

        rank = None

        for i, result in enumerate(results, 1):
            if q["source"] in result["url"]:
                rank = i
                break

        ranks.append(rank)

    metrics = {
        "Recall@1": sum(r == 1 for r in ranks) / len(ranks),
        "Recall@3": sum(r is not None and r <= 3 for r in ranks) / len(ranks),
        "Recall@5": sum(r is not None and r <= 5 for r in ranks) / len(ranks),
        "MRR": np.mean([1/r if r is not None else 0 for r in ranks])
    }

    return metrics, ranks


rerank_results, rerank_ranks = evaluate_reranker(
    questions, small_chunks, small_embeddings, small_bm25
)

print("HYBRID + CROSS-ENCODER RERANKING (200-WORD CHUNKS)")

for k, v in rerank_results.items():
    print(f"{k}: {v:.4f}")

print("\nQUESTION-WISE RANKS")

for i, rank in enumerate(rerank_ranks):
    print(f"Q{i+1}: {rank}")

HYBRID + CROSS-ENCODER RERANKING (200-WORD CHUNKS)
Recall@1: 1.0000
Recall@3: 1.0000
Recall@5: 1.0000
MRR: 1.0000

QUESTION-WISE RANKS
Q1: 1
Q2: 1
Q3: 1
Q4: 1
Q5: 1
Q6: 1
Q7: 1
Q8: 1
Q9: 1
Q10: 1
Q11: 1
Q12: 1
Q13: 1
Q14: 1
Q15: 1


In [ ]:
import json
import os

reranking_data = {
    "method": "Hybrid retrieval + Cross-Encoder",
    "chunk_size": 200,
    "candidate_k": 10,
    "final_top_k": 5,
    "metrics": {
        k: float(v) for k, v in rerank_results.items()
    },
    "question_wise_ranks": [
        {
            "question": q["question"],
            "rank": rerank_ranks[i]
        }
        for i, q in enumerate(questions)
    ]
}

path = "/content/task2_data/evaluation/reranking_metrics.json"

with open(path, "w") as f:
    json.dump(reranking_data, f, indent=4)

print("Reranking results saved successfully.")


Reranking results saved successfully.


In [ ]:
def ask_rag(question):
    results = rerank_retrieval(
        question,
        small_chunks,
        small_embeddings,
        small_bm25,
        candidate_k=10,
        top_k=5
    )

    print("QUESTION:", question)
    print("\nRETRIEVED ANSWER CONTEXT:\n")

    for i, item in enumerate(results, 1):
        print(f"PASSAGE {i}")
        print("Relevance score:", round(item["rerank_score"], 4))
        print("Source:", item["url"])
        print("Text:", item["text"][:1000])
        print("-" * 70)


ask_rag("How does a decision tree select a feature for splitting?")

QUESTION: How does a decision tree select a feature for splitting?

RETRIEVED ANSWER CONTEXT:

PASSAGE 1
Relevance score: 6.4613
Source: https://scikit-learn.org/stable/modules/tree.html
Text: vector \(y \in R^l\) , a decision tree recursively partitions the feature space such that the samples with the same labels or similar target values are grouped together. Let the data at node \(m\) be represented by \(Q_m\) with \(n_m\) samples. For each candidate split \(\theta = (j, t_m)\) consisting of a feature \(j\) and threshold \(t_m\) , partition the data into \(Q_m^{left}(\theta)\) and \(Q_m^{right}(\theta)\) subsets \[ \begin{align}\begin{aligned}Q_m^{left}(\theta) = \{(x, y) | x_j \leq t_m\}\\Q_m^{right}(\theta) = Q_m \setminus Q_m^{left}(\theta)\end{aligned}\end{align} \] The quality of a candidate split of node \(m\) is then computed using an impurity function or loss function \(H()\) , the choice of which depends on the task being solved (classification or regression) \[G(Q_m, \theta

In [ ]:
def ask_rag(question):
    results = rerank_retrieval(
        question,
        small_chunks,
        small_embeddings,
        small_bm25,
        candidate_k=10,
        top_k=5
    )

    print("QUESTION:", question)
    print("\nBEST MATCHED PASSAGE")
    print("Source:", results[0]["url"])
    print("Relevance score:", round(results[0]["rerank_score"], 4))
    print("\n", results[0]["text"])

    print("\nSUPPORTING PASSAGES")

    for i, item in enumerate(results[1:], 2):
        print(f"\nPassage {i}")
        print("Source:", item["url"])
        print(item["text"][:500])


ask_rag("How does a decision tree select a feature for splitting?")

QUESTION: How does a decision tree select a feature for splitting?

BEST MATCHED PASSAGE
Source: https://scikit-learn.org/stable/modules/tree.html
Relevance score: 6.4613

 vector \(y \in R^l\) , a decision tree recursively partitions the feature space such that the samples with the same labels or similar target values are grouped together. Let the data at node \(m\) be represented by \(Q_m\) with \(n_m\) samples. For each candidate split \(\theta = (j, t_m)\) consisting of a feature \(j\) and threshold \(t_m\) , partition the data into \(Q_m^{left}(\theta)\) and \(Q_m^{right}(\theta)\) subsets \[ \begin{align}\begin{aligned}Q_m^{left}(\theta) = \{(x, y) | x_j \leq t_m\}\\Q_m^{right}(\theta) = Q_m \setminus Q_m^{left}(\theta)\end{aligned}\end{align} \] The quality of a candidate split of node \(m\) is then computed using an impurity function or loss function \(H()\) , the choice of which depends on the task being solved (classification or regression) \[G(Q_m, \theta) = \frac{n_m^{left}

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import shutil

base_path = "/content/drive/MyDrive/tndrr_ai_ml/Task2"

os.makedirs(base_path, exist_ok=True)

print("Task 2 Google Drive folder created:")
print(base_path)

Task 2 Google Drive folder created:
/content/drive/MyDrive/tndrr_ai_ml/Task2


In [ ]:
source = "/content/task2_data"
destination = base_path + "/task2_data"

if os.path.exists(destination):
    shutil.rmtree(destination)

shutil.copytree(source, destination)

print("All Task 2 data copied successfully.")

for root, dirs, files in os.walk(destination):
    for file in files:
        print(os.path.relpath(os.path.join(root, file), destination))

All Task 2 data copied successfully.
documents.json
embeddings/large_embeddings.npy
embeddings/small_embeddings.npy
chunks/small_chunks.json
chunks/large_chunks.json
evaluation/retrieval_metrics.json
evaluation/questions.json
evaluation/reranking_metrics.json
evaluation/failure_analysis.json
evaluation/retrieval_comparison.json


In [ ]:
readme = """
# Task 2: Retrieval-Augmented Generation (RAG)

## Project Overview
Built a document retrieval and question-answering system using the official
Scikit-learn User Guide. The system retrieves relevant documentation passages
without using LangChain or LlamaIndex.

## Dataset
- Source: Official Scikit-learn User Guide
- Documents collected: 40
- Total extracted words: approximately 122,834
- Evaluation questions: 15

## Architecture
1. Collect documentation using Requests and BeautifulSoup.
2. Clean and extract document text.
3. Split documents into chunks of 200 and 500 words.
4. Generate embeddings using all-MiniLM-L6-v2.
5. Perform semantic retrieval using cosine similarity.
6. Perform keyword retrieval using BM25.
7. Combine rankings using Reciprocal Rank Fusion (RRF).
8. Rerank candidates using cross-encoder/ms-marco-MiniLM-L-6-v2.
9. Return relevant passages with source URLs.

## Evaluation
Metrics measure retrieval of the expected source documentation page
within the top results, not exact-passage accuracy.

### Semantic Retrieval
| Metric | 200 words | 500 words |
|---|---:|---:|
| Recall@1 | 93.33% | 100% |
| Recall@3 | 100% | 100% |
| Recall@5 | 100% | 100% |
| MRR | 0.9667 | 1.0000 |

### BM25 Retrieval
| Metric | 200 words | 500 words |
|---|---:|---:|
| Recall@1 | 86.67% | 80% |
| Recall@3 | 86.67% | 80% |
| Recall@5 | 100% | 93.33% |
| MRR | 0.8967 | 0.8267 |

### Hybrid Retrieval
| Metric | 200 words | 500 words |
|---|---:|---:|
| Recall@1 | 100% | 93.33% |
| Recall@3 | 100% | 100% |
| Recall@5 | 100% | 100% |
| MRR | 1.0000 | 0.9556 |

### Hybrid + Cross-Encoder Reranking
| Metric | 200 words |
|---|---:|
| Recall@1 | 100% |
| Recall@3 | 100% |
| Recall@5 | 100% |
| MRR | 1.0000 |

## Failure Analysis
1. Decision-tree queries sometimes ranked ensemble passages above the
   relevant tree passage with smaller chunks.
2. The manifold-learning page contained irrelevant lyric-like text,
   which affected semantic retrieval.
3. Some Isolation Forest queries retrieved general outlier-detection
   passages rather than the most specific explanation.

## Potential Improvements
- Heading-aware and semantic chunking.
- Improved HTML cleaning.
- Hybrid keyword and semantic retrieval.
- Cross-encoder reranking.
- Larger and more diverse evaluation datasets.
- Exact passage-level relevance annotation.

## Limitations
The evaluation contains 15 questions and uses expected source-page matching.
The reported metrics should not be interpreted as exact-passage accuracy
or general performance on unseen questions.

## Technologies
Python, Google Colab, Requests, BeautifulSoup, NumPy, Pandas,
Sentence Transformers, Scikit-learn, rank-bm25.
"""

readme_path = base_path + "/README.md"

with open(readme_path, "w") as f:
    f.write(readme)

print("Task 2 README created:", readme_path)

Task 2 README created: /content/drive/MyDrive/tndrr_ai_ml/Task2/README.md


In [ ]:
for root, dirs, files in os.walk(base_path):
    for file in files:
        path = os.path.join(root, file)
        print(file, "-", round(os.path.getsize(path) / 1024, 2), "KB")

print("\nTask 2 packaging verification completed.")

README.md - 2.62 KB
documents.json - 817.31 KB
large_embeddings.npy - 457.62 KB
small_embeddings.npy - 1173.12 KB
small_chunks.json - 1177.97 KB
large_chunks.json - 1011.95 KB
retrieval_metrics.json - 2.83 KB
questions.json - 2.09 KB
reranking_metrics.json - 2.05 KB
failure_analysis.json - 0.9 KB
retrieval_comparison.json - 1.1 KB

Task 2 packaging verification completed.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

base = "/content/drive/MyDrive/tndrr_ai_ml"

for root, dirs, files in os.walk(base):
    level = root.replace(base, "").count(os.sep)
    print("  " * level + os.path.basename(root) + "/")
    for file in files:
        print("  " * (level + 1) + file)

Mounted at /content/drive
tndrr_ai_ml/
  Final_Report.pdf
  Task2/
    README.md
    Task1/
      training_results.json
      best_lora/
        README.md
        adapter_model.safetensors
        adapter_config.json
        chat_template.jinja
        tokenizer_config.json
        tokenizer.json
    task2_data/
      documents.json
      chunks/
        large_chunks.json
        small_chunks.json
      embeddings/
        small_embeddings.npy
        large_embeddings.npy
      evaluation/
        questions.json
        retrieval_metrics.json
        failure_analysis.json
        retrieval_comparison.json
        reranking_metrics.json
  Task1_Fine_Tuning/
    training_results.json
    Task1_Fine_Tuning.ipynb
    README.md
    best_lora/
      adapter_model.safetensors
      tokenizer_config.json
      chat_template.jinja
      README.md
      adapter_config.json
      tokenizer.json
  Task2_RAG/
    README.md
    Task2_RAG.ipynb
    task2_data/
      documents.json
      chunks/
     